# 欢迎来到第 2 天实验（Day 2 Lab）！

## 今天学什么

- 什么是 **Chat Completions API**（对话补全）
- 如何用原始 HTTP（`requests`）调用端点（Endpoint）
- 如何用官方 **OpenAI Python 客户端** 做同样的事
- 如何把同一客户端指向 **Gemini / Ollama** 等 OpenAI 兼容端点
- 家庭作业：把 Day 1 网页摘要改成走本地开源模型



<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">在我们开始之前 ——</h2>
            <span style="color:#f71;">我想先介绍课程的资源页：里面有全部幻灯片链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            建议收藏；作者会持续补充有用链接。
            </span>
        </td>
    </tr>
</table>



## 首先 —— 聊聊 Chat Completions API

1. 这是调用 LLM 最简单、最常见的方式之一
2. 之所以叫「聊天补全（Chat Completions）」，可以理解为：给定一段对话，请模型预测「下一句助手会说什么」
3. 该 API 由 OpenAI 推广开来，生态上非常流行，很多厂商都做了兼容实现

### 我们会先再次调用 OpenAI

不用担心「我不用 OpenAI」——后面马上会讲 Gemini、本地 Ollama 等兼容端点。



In [ ]:
# ========== 环境检查：从 .env 读取 OPENAI_API_KEY ==========

# 导入标准库 os：通过环境变量读取密钥
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件加载进进程环境
from dotenv import load_dotenv

# override=True：即使环境里已有同名变量，也用 .env 覆盖（便于笔记本里改密钥后立刻生效）
load_dotenv(override=True)
# 读取 OpenAI 密钥；变量名必须是 OPENAI_API_KEY（和常见 SDK 约定一致）
api_key = os.getenv('OPENAI_API_KEY')

# 三分支自检：缺失 / 前缀不像 sk-proj- / 看起来正常（打印文案保持英文原样，便于对照排查指南）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")



## 你知道什么是端点（Endpoint）吗？

如果还不熟，请先看本仓库 Guides 文件夹里的技术基础指南。

下面这几格会直接对 OpenAI 的 Chat Completions **HTTP 端点** 发请求——先不用 SDK，看清底层在干什么。



In [ ]:
# ========== 准备 HTTP 请求：headers + JSON payload ==========

# 导入 requests：用 Python 发 HTTP POST
import requests

# Authorization：Bearer + API Key；Content-Type 声明正文是 JSON
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

# payload：Chat Completions 请求体（model / messages）；模型 id 与 prompt 字符串保持原样
payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

# 在笔记本里直接显示 payload，方便确认结构
payload



In [ ]:
# ========== 原始 HTTP：POST 到 /v1/chat/completions ==========

# 对官方端点发 POST：headers 带鉴权，json=payload 自动序列化并设正文
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

# 把响应体解析成 Python 字典/列表并展示（完整 JSON，含 usage 等字段）
response.json()



In [ ]:
# ========== 从响应 JSON 里取出助手回复正文 ==========

# choices[0].message.content：第一条候选回复的文本内容
response.json()["choices"][0]["message"]["content"]



# `openai` 包是什么？

它是 **Python 客户端库（client library）**。

本质上就是对 HTTP 端点的一层封装：让你用更干净的 Python 调用，而不必手写 headers / JSON。

它是开源、轻量的；**并不包含** OpenAI 模型权重本身——只是帮你发请求、解析响应。



In [ ]:
# ========== 用 OpenAI 官方客户端做同样的 Chat Completions ==========

# 从 openai 导入 OpenAI 类
from openai import OpenAI
# 无参构造：默认从环境变量 OPENAI_API_KEY 读密钥，base_url 指向官方 API
openai = OpenAI()

# 一次 create：等价于前面的 HTTP POST，但返回结构化对象而不是裸 JSON
response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 取出助手回复文本（属性访问，而不是字典下标）
response.choices[0].message.content



## 然后发生了一件大事

OpenAI 的 Chat Completions API 太流行了，其他模型厂商也做出了**相同形态**的端点——称为 **OpenAI 兼容端点（OpenAI-compatible endpoints）**。

例如 Google 提供了：https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI 也允许你在同一套客户端里指定不同的 `base_url` 与 `api_key`，去调用其他提供商。

所以你可以这样写（示例，密钥请换成你自己的）：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

要强调：代码里虽然写着 `OpenAI(...)`，这里用的只是轻量客户端去打端点——**并不等于**在跑 OpenAI 的模型。

若仍困惑，请看 Guides 文件夹里的 Guide 9。

## 可选：试用 Google Gemini

1. 打开 https://aistudio.google.com/
2. 在 https://aistudio.google.com/api-keys 创建 API Key
3. 写入 `.env` 并保存：

`GOOGLE_API_KEY=AIz...`



In [ ]:
# ========== 检查 Google Gemini 的 API Key ==========

# Gemini 的 OpenAI 兼容 base_url（字符串必须保持原样）
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 再次加载 .env，确保刚写入的 GOOGLE_API_KEY 能读到
load_dotenv(override=True)

# 从环境变量读取 Google API Key
google_api_key = os.getenv("GOOGLE_API_KEY")

# 自检：缺失 / 前缀不像 AIz / 看起来正常（英文提示保持原样）
if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
# ========== 用同一 OpenAI 客户端调用 Gemini ==========

# base_url + api_key 指向 Google；变量名 gemini 只是客户端实例名
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

# 模型 id 为 Gemini 侧名称；messages 写法与 OpenAI 相同
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 取出文本回复
response.choices[0].message.content



## Ollama 也提供 OpenAI 兼容端点

……而且跑在你的**本地电脑**上！

如果下一格没有打印出类似 `Ollama is running` 的内容，请打开终端运行 `ollama serve`（有的环境脚本写作 `ollamaserve`）。



In [ ]:
# ========== 探测本地 Ollama 是否在跑 ==========

# GET 本地根地址；若服务正常，响应体常见为 b'Ollama is running'
requests.get("http://localhost:11434").content



### 从 Meta 生态下载 `llama3.2`

若机器内存较小，可改用 `llama3.2:1b`。

不要轻易上 `llama3.3` 或 `llama4`——对很多笔记本来说太大了。



In [ ]:
# 在笔记本里执行 shell：拉取本地模型权重（需本机已安装 ollama CLI）
!ollama pull llama3.2:1b



In [ ]:
# ========== 创建指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# 再次导入 OpenAI（本格可独立运行时更直观）
from openai import OpenAI
# Ollama 的 OpenAI 兼容基址：注意是 /v1
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；这里用占位 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')



In [ ]:
# ========== 用本地 llama3.2:1b 要一个 fun fact ==========

# model 名必须与已 pull 的一致；messages 与云端写法相同
response = ollama.chat.completions.create(model="llama3.2:1b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 打印助手回复
response.choices[0].message.content



In [ ]:
# 再试 deepseek-r1:1.5b（注释说明：这是从阿里云侧「蒸馏」到 Qwen 系列相关的 DeepSeek 小模型）
# 先 pull，下一格再调用
!ollama pull deepseek-r1:1.5b



In [ ]:
# ========== 用本地 deepseek-r1:1.5b 再问一次 ==========

# 仍然走同一个 ollama 客户端，只换 model 字符串
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 取出回复（推理模型可能会带思考痕迹，以实际输出为准）
response.choices[0].message.content



# 家庭作业练习

把第 1 天的「网页摘要」项目升级为：通过 **Ollama** 在本地跑开源模型，而不是调用付费 OpenAI。

若你不想用付费 API，后续很多项目都可以复用这套「OpenAI 兼容 + 本地 base_url」技术。

**好处：**
1. 无按次 API 费用——模型开源、本地推理
2. 数据不必离开你的电脑

**缺点：**
1. 能力通常明显弱于 Frontier（前沿）云端大模型

## Ollama 安装回顾

1. 打开 [ollama.com](https://ollama.com) 安装
2. 安装后本地服务一般会自动运行；访问 [http://localhost:11434/](http://localhost:11434/) 应看到 “Ollama is running”
3. 若没有：新开终端运行 `ollama serve`，另开终端 `ollama pull llama3.2`，再刷新上述地址
4. 若太慢：改用 `llama3.2:1b`（`ollama pull llama3.2:1b`），并把代码里的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`

下面单元格是一份变体作业：抓 LinkedIn 公开页线索 → 解析字段 → 让本地模型写简短介绍邮件（提示词保持英文）。



In [ ]:
# ========== 作业依赖：展示工具 + HTTP + OpenAI 客户端 + JSON ==========

# Markdown/display：把模型输出渲染成笔记本文本
from IPython.display import Markdown, display
# requests：后面探测 Ollama、抓网页都会用到
import requests
# OpenAI：连接本地 Ollama 的兼容端点
from openai import OpenAI
# json：把结构化 profile 字典漂亮地序列化进 user prompt
import json



In [ ]:
# ========== 再次确认 Ollama 本地服务可用 ==========

# 与前面探测相同；作业段可独立重跑时更方便
requests.get("http://localhost:11434").content



In [ ]:
# ========== system prompt：规定模型扮演「职业社交助手」 ==========

# 发给模型的指令保持英文，避免改变语气与任务边界
system_prompt = """
You are a professional networking assistant.

Your task is to write a short professional introduction email based on profile details.
"""



In [ ]:
# ========== user prompt 前缀：说明后面会附上网站/画像内容 ==========

# 真正的结构化数据会在 messages_for 里拼到这个前缀后面
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.

"""



In [ ]:
# ========== 组装 Chat Completions 的 messages 列表 ==========

def messages_for(website):
    return [
        # system：角色与任务（写简短介绍邮件）
        {"role": "system", "content": system_prompt},
        # user：前缀 + 把 website（可为 dict）格式化成缩进 JSON 字符串
        {"role": "user", "content": user_prompt_prefix +  json.dumps(website, indent=2)}
    ]



In [ ]:
# ========== 抓取与解析 LinkedIn 公开页（简易版，逻辑保持原样） ==========

# 导入 requests：HTTP GET
import requests
# 导入 BeautifulSoup：HTML 解析
from bs4 import BeautifulSoup

def scrape_linkedin_profile(url):
    """请求页面，抽出 title / meta description / 可见文本预览。"""

    # 浏览器风格 UA，降低被简单反爬直接拦下的概率
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    # 发起 GET
    response = requests.get(url, headers=headers)

    # 非 200：返回错误字典（字符串保持原样，可能被后续逻辑读到）
    if response.status_code != 200:
        return {"error": "Unable to fetch page"}

    # 解析 HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # 用字典累积字段
    profile = {}

    # 提取 <title> 文本
    title = soup.title.text if soup.title else ""
    profile["title"] = title

    # 提取 meta description（公开页常含一句话简介）
    description_tag = soup.find("meta", {"name": "description"})
    if description_tag:
        profile["headline"] = description_tag.get("content")

    # 去掉 script/style，再取可见文本
    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text("\n", strip=True)

    # 只保留前 1500 字符，控制上下文体积
    profile["preview_text"] = text[:1500]

    return profile


def parse_profile(data):
    """从 title/headline 字符串里启发式抽出姓名、公司、教育、地点等字段。"""

    title = data.get("title", "")
    headline = data.get("headline", "")

    profile = {}

    # title 常见形态：Name - Company | LinkedIn
    parts = title.replace("| LinkedIn","").split("-")

    profile["name"] = parts[0].strip() if len(parts) > 0 else ""
    profile["company"] = parts[1].strip() if len(parts) > 1 else ""

    # headline 里用 · 分隔的片段再细分
    items = headline.split("·")

    for item in items:
        item = item.strip()

        # Experience: 后面当作公司信息覆盖
        if "Experience:" in item:
            profile["company"] = item.replace("Experience:", "").strip()

        # Education: 教育信息
        elif "Education:" in item:
            profile["education"] = item.replace("Education:", "").strip()

        # Location: 地点
        elif "Location:" in item:
            profile["location"] = item.replace("Location:", "").strip()

        # 含 years 的片段当作年限/经验描述
        elif "years" in item.lower():
            profile["experience"] = item

    return profile



In [ ]:
# ========== summarize：抓取 → 解析 → 本地 LLM 生成 ==========

def summarize(url):
    # 本地 Ollama 的 OpenAI 兼容地址
    OLLAMA_BASE_URL = "http://localhost:11434/v1"
    # 每次调用时新建客户端（逻辑保持原样）
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    # 抓公开页原始字段
    website = scrape_linkedin_profile(url)
    # 启发式解析成更干净的 dict
    data = parse_profile(website)
    # 用本地小模型生成（提示词组合来自 messages_for）
    response = ollama.chat.completions.create(
        model = "llama3.2:1b",
        messages = messages_for(data)
    )
    # 返回助手文本
    return response.choices[0].message.content



In [ ]:
# ========== display_summary：生成并用 Markdown 展示 ==========

def display_summary(url):
    # 调用上一格的 summarize
    summary = summarize(url)
    # 在笔记本输出区渲染 Markdown
    display(Markdown(summary))



In [ ]:
# ========== 端到端演示：对指定 LinkedIn URL 生成介绍邮件 ==========

# URL 保持原样；若页面需登录/反爬，抓取结果可能很稀疏，属预期现象
display_summary("https://www.linkedin.com/in/neha-bhoi-52901994/")

